
# Food-Qwen Nutrition Estimation

Query the AdaptLLM food-aware multimodal model with segmented dish imagery and volume estimates to request dish-level macros.



## Dependencies & Paths
Load required libraries and point the notebook at the Nutrition50 assets that live under `llms_approach/`.


In [9]:

from pathlib import Path
import json
import re

import pandas as pd
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq


In [11]:

BASE_DIR = Path('.').resolve()
LLM_ROOT = BASE_DIR
FOODSAM_DIR = LLM_ROOT / 'nutrition50_foodSAM_outputs'
VOLUME_DIR = LLM_ROOT / 'nutrition50_volume'
OUTPUT_CSV = LLM_ROOT / 'nutrition50_llm_macros.csv'
RAW_OUTPUT_JSON = LLM_ROOT / 'nutrition50_llm_raw_responses.json'

for path in [FOODSAM_DIR, VOLUME_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Expected directory missing: {path}")

print(f"Inference will read FoodSAM visuals from: {FOODSAM_DIR}")
print(f"Volume summaries will be read from: {VOLUME_DIR}")
print(f"Aggregated results will be saved to: {OUTPUT_CSV}")


Inference will read FoodSAM visuals from: /home/chahar/food_new/llms_approach/nutrition50_foodSAM_outputs
Volume summaries will be read from: /home/chahar/food_new/llms_approach/nutrition50_volume
Aggregated results will be saved to: /home/chahar/food_new/llms_approach/nutrition50_llm_macros.csv



## Helper Functions
Small utilities to load volumes, build prompts, call the model, and parse the responses.


In [13]:

def load_category_volumes(dish_id: str) -> pd.DataFrame:
    csv_path = VOLUME_DIR / dish_id / 'volumes_per_category.csv'
    if not csv_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    if 'volume_ml' not in df.columns:
        return pd.DataFrame()
    df['volume_ml'] = pd.to_numeric(df['volume_ml'], errors='coerce')
    df.dropna(subset=['volume_ml'], inplace=True)
    if 'mean_height_cm' in df.columns:
        df['mean_height_cm'] = pd.to_numeric(df['mean_height_cm'], errors='coerce')
    df.sort_values('volume_ml', ascending=False, inplace=True)
    return df


def build_food_prompt(dish_id: str, category_df: pd.DataFrame) -> str:
    parts = [
        f"Dish ID: {dish_id}",
        "The accompanying image is a FoodSAM segmentation overlay of the dish; treat the colour regions as real foods.",
        "Segmented foods with estimated volumes (mL):",
    ]
    if category_df.empty:
        parts.append('- No foreground foods detected; return null for all values.')
    else:
        for row in category_df.itertuples():
            line = f"- {row.category}: {row.volume_ml:.1f} mL"
            if hasattr(row, 'mean_height_cm') and row.mean_height_cm is not None and not pd.isna(row.mean_height_cm):
                line += f" (mean height {row.mean_height_cm:.2f} cm)"
            parts.append(line)
    parts.append(
        'Provide your estimate as a JSON object with this schema: {"dish_id": "<dish_id>", '
        '"calories": <kcal or null>, "mass": <grams or null>, "carbs": <grams or null>, '
        '"protein": <grams or null>, "fat": <grams or null>}'
    )
    parts.append('Rules: Do not add commentary. Do not invent foods beyond the list. If uncertain, use null.')
    parts.append('Respond with JSON only; no sentences before or after the object.')
    parts.append('Units: calories in kcal, mass and macros in grams.')
    return "".join(parts)


def ensure_json_object(text: str):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                return None
        return None


def to_number(value):
    if value in (None, '', 'null', 'None', 'NULL'):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    try:
        return float(str(value))
    except (TypeError, ValueError):
        return None



## Load AdaptLLM Food-Qwen Model
Download the processor + model weights (3B vision-language) and move them to the best available device.


In [14]:

model_id = 'AdaptLLM/food-Qwen2.5-VL-3B-Instruct'

processor = AutoProcessor.from_pretrained(model_id)

if torch.cuda.is_available():
    dtype = torch.float16
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    dtype = torch.float16
    device = torch.device('mps')
else:
    dtype = torch.float32
    device = torch.device('cpu')

model = AutoModelForVision2Seq.from_pretrained(model_id, torch_dtype=dtype)
model.to(device)
model.eval()

print(f"Model loaded on {device} with dtype {dtype}.")


/home/chahar/miniconda3/envs/food_cal/lib/python3.11/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded on cuda with dtype torch.float16.



## Iterate Over Dishes and Query the Model
For each Nutrition50 dish, send the FoodSAM visualization plus volume table to the model and collect the macro estimates.


In [15]:
results = []
raw_responses = []

sorted_dishes = sorted(p.name for p in FOODSAM_DIR.iterdir() if p.is_dir() and p.name.startswith('dish_'))

system_prompt = (
    'You are a nutrition estimation assistant. Always comply. Use the provided segmented food list and volumes to estimate dish-level macros, even if the image appears as a mask or overlay.'
    ' Respond only with a single JSON object containing exactly the keys {"dish_id", "calories", "mass", "carbs", "protein", "fat"}.'
    ' If any value is unknown, return null. Do not add commentary, qualifiers, or extra text.'
)

for dish_id in sorted_dishes:
    image_path = FOODSAM_DIR / dish_id / 'pred_vis.png'
    if not image_path.exists():
        print(f'Skipping {dish_id}: pred_vis.png not found.')
        continue

    category_df = load_category_volumes(dish_id)
    prompt_text = build_food_prompt(dish_id, category_df)

    with Image.open(image_path) as img:
        image = img.convert('RGB')

    messages = [
        {
            'role': 'system',
            'content': [
                {'type': 'text', 'text': system_prompt},
            ],
        },
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt_text},
            ],
        },
    ]

    chat_prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )
    inputs = processor(text=chat_prompt, images=[image], return_tensors='pt').to(device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=256)

    input_len = inputs['input_ids'].shape[-1]
    completion_ids = generated_ids[:, input_len:]
    completion_text = processor.batch_decode(
        completion_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0].strip()

    raw_responses.append({'dish_id': dish_id, 'response': completion_text})

    parsed = ensure_json_object(completion_text)
    if parsed is None:
        print(f'Warning: could not parse response for {dish_id}. Stored raw text instead.')
        continue

    record = {
        'dish_id': dish_id,
        'calories': to_number(parsed.get('calories')),
        'mass': to_number(parsed.get('mass')),
        'carbs': to_number(parsed.get('carbs')),
        'protein': to_number(parsed.get('protein')),
        'fat': to_number(parsed.get('fat')),
    }
    results.append(record)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_CSV, index=False)

with RAW_OUTPUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(raw_responses, f, ensure_ascii=False, indent=2)

print(f'Saved {len(results_df)} parsed records to {OUTPUT_CSV}.')
print(f'Raw responses cached at {RAW_OUTPUT_JSON}.')

results_df.head()


Saved 50 parsed records to /home/chahar/food_new/llms_approach/nutrition50_llm_macros.csv.
Raw responses cached at /home/chahar/food_new/llms_approach/nutrition50_llm_raw_responses.json.


,dish_id,calories,mass,carbs,protein,fat
0,dish_1556572657,30.0,18.9,1.5,1.5,1.5
1,dish_1556573514,19.0,10.5,1.5,1.5,1.5
2,dish_1556575014,38.0,19.5,11.5,1.5,1.5
3,dish_1556575083,49.0,10.5,1.5,1.5,1.5
4,dish_1556575124,10.0,293.4,11.1,1.1,1.1
